# Proyecto Final: Sistema de Recomendación Híbrido de Laptops

Este notebook presenta la fundamentación matemática y la implementación paso a paso de un **Sistema de Recomendación Híbrido de Laptops** en dos niveles:
1. **Filtro en Cascada (Cascade Filtering)**: Usando reglas de inferencia lógica basadas en restricciones duras de hardware.
2. **Ensamble Ponderado Dinámico (Dynamic Weighted Ensemble)**: Combinando tres modelos complementarios (MAUT, SVD y Similaridad Coseno) con coeficientes adaptables según el perfil de usuario.

## 🧩 Los 4 Sub-Modelos del Sistema Híbrido

### 🟢 Modelo 1: Motor de Inferencia Lógica (Constraint-Based)
Evalúa reglas duras (vías máscara binaria $M_{rules}(i) \in \{0, 1\}$) para descartar opciones técnicamente inviables según el caso de uso y el presupuesto.
$$M_{rules}(i) = \begin{cases} 1 & \text{si cumple todas las reglas duras} \\ 0 & \text{si viola al menos una regla} \end{cases}$$

### 🔵 Modelo 2: Multi-Attribute Utility Theory (MAUT) Score (Knowledge-Based)
Normaliza las especificaciones (precio, rendimiento, portabilidad, pantalla) y aplica una función de utilidad ponderada por sliders:
$$U_{Knowledge}(i) = w_{price} \cdot S_{price}(i) + w_{perf} \cdot S_{perf}(i) + w_{portability} \cdot S_{portability}(i) + w_{screen} \cdot S_{screen}(i)$$

### 🟣 Modelo 3: Matrix Factorization mediante SVD (Collaborative Filtering)
Predice valoraciones implícitas basadas en la comunidad de usuarios similares usando descomposición de factores latentes optimizada con SGD:
$$\hat{r}_{u,i} = \mu + b_u + b_i + P_u^T Q_i$$

### 🔴 Modelo 4: Similitud Coseno de Atributos (Content-Based Vector Embedding)
Vectoriza las especificaciones técnicas ($v_i$) y calcula el ángulo del coseno respecto al vector de perfil del usuario ($v_u$):
$$S_{content}(i) = \cos(v_u, v_i) = \frac{v_u \cdot v_i}{\|v_u\| \|v_i\|}$$

---

## 1. Configuración del Entorno de Ejecución
Agregamos el directorio raíz del proyecto al `sys.path` para importar correctamente los módulos locales.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Agregar directorio padre al path
parent_dir = os.path.dirname(os.path.abspath(''))
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

print("Directorio raíz configurado:", parent_dir)

## 2. Inicialización de Datos
Cargamos los DataFrames de laptops y calificaciones de usuarios. Si no existen, los generamos de forma sintética.

In [ ]:
from src.data_generator import generate_datasets

laptops_path = "../data/laptops.csv"
ratings_path = "../data/ratings.csv"

if not os.path.exists(laptops_path) or not os.path.exists(ratings_path):
    print("Generando nuevos datasets sintéticos...")
    generate_datasets(dest_dir="../data")
else:
    print("Datasets encontrados en la ruta especificada.")

laptops_df = pd.read_csv(laptops_path)
ratings_df = pd.read_csv(ratings_path)

print(f"Total Laptops: {len(laptops_df)}")
print(f"Total Calificaciones (Interacciones): {len(ratings_df)}")

### Vista Previa del Catálogo de Laptops

In [ ]:
laptops_df.head(3)

### Vista Previa de la Matriz de Ratings de Usuarios

In [ ]:
ratings_df.head(5)

## 3. Pruebas del Modelo 1: Motor de Inferencia Lógica
Evaluamos el funcionamiento de las reglas declarativas sobre un perfil demandante: `Deep Learning` (VRAM >= 6GB, RAM >= 16GB, CUDA = True) con un presupuesto de 1500 USD.

In [ ]:
from src.rules_engine import LogicInferenceModel

rules_model = LogicInferenceModel("../data/knowledge_rules.json")

usage = "Deep Learning"
budget = 1500.0

mask = rules_model.get_binary_mask(laptops_df, usage, budget)
print(f"Laptops viables de hardware y presupuesto: {sum(mask)} de {len(laptops_df)}")

# Mostrar las laptops que pasaron las reglas duras
laptops_df[mask == 1][["brand", "name", "price", "ram", "gpu_vram", "cuda_support"]].head(5)

## 4. Pruebas del Modelo 2: MAUT Utility Score
Calculamos el puntaje de utilidad multi-atributo para las laptops basándonos en sliders de preferencias cualitativas del usuario (e.g. prioridad en precio bajo y rendimiento).

In [ ]:
from src.maut_model import MAUTModel

maut_model = MAUTModel()
maut_model.fit(laptops_df)

# Pesos asignados por el usuario (Precio y Rendimiento alta prioridad)
weights = {
    "price": 0.4,
    "perf": 0.4,
    "portability": 0.1,
    "screen": 0.1
}

utility_scores = maut_model.compute_utility(laptops_df, weights)
laptops_df_u = laptops_df.copy()
laptops_df_u["maut_utility"] = utility_scores

print("Top 5 Laptops con mayor utilidad MAUT:")
laptops_df_u.sort_values(by="maut_utility", ascending=False)[["name", "price", "ram", "gpu_vram", "weight", "maut_utility"]].head(5)

## 5. Pruebas del Modelo 3: Matrix Factorization SVD
Entrenamos el SVD colaborativo usando SGD en numpy y predecimos la valoración implícita del usuario U001 en laptop con ID 5.

In [ ]:
from src.svd_model import MatrixFactorizationSVD

svd_model = MatrixFactorizationSVD()
svd_model.fit(ratings_df)

user_id = "U001"
laptop_id = 5

rating_pred = svd_model.predict(user_id, laptop_id)
rating_pred_norm = svd_model.predict_normalized(user_id, laptop_id)

print(f"Predicción de rating para el usuario {user_id} e item {laptop_id}:")
print(f"Rating original (1-5 estrellas): {rating_pred:.2f}★")
print(f"Rating normalizado (0-1): {rating_pred_norm:.4f}")


## 6. Pruebas del Modelo 4: Similitud Coseno de Atributos (Content-Based)
Calculamos la similitud coseno entre las laptops y un vector ideal teórico del usuario según sus preferencias de hardware y portabilidad.

In [ ]:
from src.content_model import ContentBasedModel

content_model = ContentBasedModel()
content_model.fit(laptops_df)

cosine_similarities = content_model.compute_similarity(laptops_df, "Deep Learning", 1500.0, weights)
laptops_df_c = laptops_df.copy()
laptops_df_c["cosine_similarity"] = cosine_similarities

print("Top 5 Laptops con mayor similitud de contenido a la ideal del usuario:")
laptops_df_c.sort_values(by="cosine_similarity", ascending=False)[["name", "price", "ram", "gpu_vram", "weight", "cosine_similarity"]].head(5)

## 7. Pruebas del Ensamble Híbrido Final
Instanciamos el recomendador híbrido global y evaluamos dos perfiles diferentes para demostrar la **Ponderación Adaptable Dinámica**:
1. **Usuario Nuevo (Cold Start)**: Da prioridad a MAUT y Contenido, bajando el peso de SVD.
2. **Usuario Recurrente**: Da prioridad al SVD colaborativo.

In [ ]:
from src.hybrid_recommender import HybridRecommender

hybrid_rec = HybridRecommender(data_dir="../data")
hybrid_rec.load_data_and_train()

user_known = "U001"
user_new = "U999"
usage = "Deep Learning"
budget = 1600.0
weights = {"price": 0.2, "perf": 0.6, "portability": 0.1, "screen": 0.1}

In [ ]:
# 1. Caso Cold Start (Usuario Nuevo)
recs_new, meta_new = hybrid_rec.get_recommendations(user_new, usage, budget, weights, top_k=5, is_cold_start=True)
print("=== RECOMENDACIONES COLD START (NUEVO USUARIO) ===")
print(f"Perfil aplicado: {meta_new['profile_type']}")
print(f"Pesos: {meta_new['weights']}\n")
print(recs_new[["name", "price", "u_knowledge", "s_content", "svd_score_norm", "hybrid_score"]])

In [ ]:
# 2. Caso Usuario Frecuente (Colaborativo)
recs_known, meta_known = hybrid_rec.get_recommendations(user_known, usage, budget, weights, top_k=5, is_cold_start=False)
print("=== RECOMENDACIONES USUARIO CON HISTORIAL ===")
print(f"Perfil aplicado: {meta_known['profile_type']}")
print(f"Pesos: {meta_known['weights']}\n")
print(recs_known[["name", "price", "u_knowledge", "s_content", "svd_score_norm", "hybrid_score"]])

## 8. Evaluación Científica del Sistema
Ejecutamos la rutina de evaluación científica y cargamos los gráficos comparativos de rendimiento. 
Esto nos permite ver el **Constraint Satisfaction Rate (CSR)** y **Precision/Recall** frente a baselines independientes.

In [ ]:
from src.evaluation import evaluate_metrics

metrics_results = evaluate_metrics(data_dir="../data", output_plot_dir="../app/static/plots")

print("\n--- RESULTADOS FINALES DE EVALUACIÓN ---")
for key, val in metrics_results.items():
    print(f"{key}: {val:.4f}")

### Visualización de Gráficos de Evaluación

In [ ]:
from IPython.display import Image, display

print("1. Tasa de Satisfacción de Restricciones (CSR@10):")
display(Image(filename="../app/static/plots/evaluation_csr.png"))

print("\n2. Precisión y Recall Comparativo vs Baseline:")
display(Image(filename="../app/static/plots/evaluation_precision_recall.png"))